In [ ]:
import pymupdf
import re
import spacy
import csv
import os

In [ ]:
DATA_FOLDER = "../data/"
PDF_FOLDER = f"{DATA_FOLDER}pdf_data/"
TXT_FOLDER = f"{DATA_FOLDER}txt_data/"
TXT_BLOCKS_FOLDER = f"{DATA_FOLDER}txt_blocks/"
CSV_OUTPUT_FILE = f"{DATA_FOLDER}actors_per_paragraph_per_year.csv"
CSV_ACTORS_MAPPING = f"{DATA_FOLDER}actors_mapping.csv"

# Load English tokenizer, tagger, parser and NER
nlp_model = spacy.load("en_core_web_sm")

In [ ]:
def extract_blocks_from_pdf(file_name: str):
    print(f"Extracting blocks from {PDF_FOLDER}{file_name}.pdf...")
    doc = pymupdf.open(f"{PDF_FOLDER}{file_name}.pdf") # open a document
    out = open(f"{TXT_BLOCKS_FOLDER}{file_name}.txt", "w", encoding="utf-8") # create a text output
    for page in doc: # iterate the document pages
        for block in page.get_text("blocks", sort=False):
            text = block[4]
            text = text.replace("\n", " ")
            text = re.sub(r"\s+", " ", text).strip()
            out.write(text + "\n\n")
    print(f"Blocks extracted and saved to {TXT_BLOCKS_FOLDER}{file_name}.txt.") 
    out.close()
    doc.close()

In [ ]:
def clear_text(file_name: str):
    file_path = f"{TXT_BLOCKS_FOLDER}{file_name}.txt"
    print(f"Cleaning text in {file_path}.txt...")

    with open(file_path, "r") as file:
        text = file.read()

    paragraphs = text.split("\n\n")

    new_paragraphs = []

    for p in paragraphs:
        if (
            "This issue of the Earth Negotiations Bulletin (ENB)" not in p 
            and len(p) > 40
            ):
            new_paragraphs.append(p)

    cleaned_text = "\n\n".join(new_paragraphs)
    
    with open(file_path, "w") as file:
        file.write(cleaned_text)
        
    print(f"Text cleaned and saved to {file_path}.")

In [ ]:
def get_actors_in_paragraph(paragraph: str) -> set:
    temp_actors = set()
    actors = set()

    # Fill temp_actors with Spacy EntityRecognizer (NER)
    # doc = nlp_model(paragraph)
    # for ent in doc.ents:
    #     if ent.label_ in ["GPE", "NORP", "ORG"]:
    #         temp_actors.add((ent.text))

    # Fill temp_actors with actors_mapping.csv
    with open(CSV_ACTORS_MAPPING, mode ='r')as file:
        csvFile = csv.reader(file)
        for lines in csvFile:
                if lines[0] in paragraph:
                    if lines[0].upper() == "CHINA" and "G-77/CHINA":
                        continue
                    if lines[0].upper() == "G-77" and "G-77/CHINA":
                        continue
                    temp_actors.add(lines[1])

    # Cleaning and mapping
    for actor in temp_actors:
        with open(CSV_ACTORS_MAPPING, mode ='r')as file:
            csvFile = csv.reader(file)
            for lines in csvFile:
                actor = actor.upper()
                if lines[0].upper() == actor or lines[1].upper() == actor:
                    actors.add(lines[1])

    return actors

In [ ]:
def get_actors_from_txt_blocks(file_name: str) -> list[set]:
    file_path = f"{TXT_BLOCKS_FOLDER}{file_name}.txt"
    print(f"Extracting actors from {file_path}...")
    actors = []

    with open(file_path, "r") as file:
        text = file.read()

    paragraphs = text.split("\n\n")

    for p in paragraphs:
        actors_per_paragraph = get_actors_in_paragraph(p)
        actors.append(actors_per_paragraph)

    print(f"Actors extracted from {file_path}.")
    return actors

In [ ]:
def build_actors_mapping_csv_file_from_txt_blocks(file_name: str):
    print(f"Building actors mapping CSV file from {TXT_BLOCKS_FOLDER}{file_name}.txt...")
    with open(CSV_OUTPUT_FILE, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        actors_per_paragraph = get_actors_from_txt_blocks(file_name)

        for index, value in enumerate(actors_per_paragraph):
            if value == set():
                continue
            actors_string = ", ".join(sorted(value))
            writer.writerow([file_name, index + 1, actors_string])
    print(f"Actors mapping CSV file built and saved to {CSV_OUTPUT_FILE} for {file_name}.txt.")

In [ ]:
with open(CSV_OUTPUT_FILE, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["year", "paragraph_id", "actors"])

for pdf in os.listdir(PDF_FOLDER):
    file_name = os.path.basename(pdf)
    print(f"Found file: {file_name}")
    if file_name == ".DS_Store":
        continue

    FILE_NAME = file_name.split(".")[0]
    print(f"Processing {FILE_NAME}...")
    
    extract_blocks_from_pdf(FILE_NAME)
    clear_text(FILE_NAME)
    build_actors_mapping_csv_file_from_txt_blocks(FILE_NAME)

    print(f"{FILE_NAME} processed!\n")
    IS_TITLE_WRITTEN = True